# 06.5 Dynamic topic modeling con BERTopic (Híbrido)

Este cuaderno prepara un flujo para modelar temas dinámicos sobre los posteos de Revista Anfibia usando una arquitectura híbrida:
- **Embeddings/Clustering:** Usa el texto original para capturar la sintaxis y semántica.
- **Representación (Palabras clave):** Usa el texto lematizado para evitar redundancia morfológica en español.

In [5]:
# EJECUTAR ESTA CELDA PRIMERO QUE NADA (Y REINICIAR EL KERNEL ANTES SI FALLÓ)
import coverage.types as ct

# Parche de compatibilidad para evitar que Numba/UMAP crasheen
for name, fallback in {
    'Tracer': getattr(ct, 'TTracer', object),
    'TShouldTraceFn': object,
    'TShouldStartContextFn': object,
    'TFileDisposition': object,
}.items():
    if not hasattr(ct, name):
        setattr(ct, name, fallback)

## 1. Configuración del entorno

In [6]:
import os
import re
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer
from bertopic.representation import KeyBERTInspired
from bertopic import BERTopic

warnings.filterwarnings("ignore", category=UserWarning)

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

# Rutas
DATA_PATH = PROJECT_ROOT / "data" / "processed" / "corpus_anfibia_lematizado.csv"
assert DATA_PATH.exists(), f"Falta archivo de entrada: {DATA_PATH.name}"

OUTPUT_DIR = PROJECT_ROOT / "outputs" / "bertopic_hibrido"
GRAFICOS_DIR = OUTPUT_DIR / "graficos"
OUTPUT_DIR.mkdir(exist_ok=True, parents=True)
GRAFICOS_DIR.mkdir(exist_ok=True, parents=True)

## 2. Carga y normalización de metadatos

In [7]:
MAX_DOCUMENTS = 4000
USE_COLUMNS = ["ID", "source_url", "title_text", "tagline", "autor_names", "post_date", "category", "text_type", "document_text_original", "document_text_final"]

df = pd.read_csv(DATA_PATH, usecols=USE_COLUMNS, nrows=MAX_DOCUMENTS)

# Llenar vacíos
df["document_text_original"] = df["document_text_original"].fillna("").astype(str).str.strip()
df["document_text_final"] = df["document_text_final"].fillna("").astype(str).str.strip()

# Filtrar vacíos
df = df[df["document_text_original"].ne("") & df["document_text_final"].ne("")].copy()
df = df.reset_index(drop=True)
df.insert(0, "doc_id", df.index)

print(f"Documentos listos: {df.shape[0]}")

# Normalizar categorías
def split_categories(value):
    parts = re.split(r"[;,]", str(value))
    return [part.strip() for part in parts if part.strip()]

category_rows = []
for doc_id, value in zip(df["doc_id"], df["category"]):
    categories = split_categories(value)
    if not categories:
        categories = ["Sin categoria"]
    for category in categories:
        category_rows.append({"doc_id": doc_id, "category_normalized": category})

categories_df = pd.DataFrame(category_rows)
category_counts = categories_df["category_normalized"].value_counts().rename_axis("category").reset_index(name="documents")

Documentos listos: 3556


## 3. Preparación de Stopwords y Textos

Definimos las palabras vacías generales del español y las específicas del dominio periodístico de Revista Anfibia. 
Separamos explícitamente el corpus en dos listas: una con el texto original (para la semántica) y otra con el texto lematizado (para la frecuencia de palabras).

In [8]:
SPANISH_STOPWORDS = {
    "a", "acá", "ahi", "ahí", "al", "algo", "algunas", "algunos", "ante", "antes", "aquel", "aquella", "aquellas", "aquello", "aquellos",
    "aqui", "aquí", "arriba", "asi", "así", "atras", "aun", "aunque", "bajo", "bien", "cada", "casi", "como", "con", "contra",
    "cual", "cuando", "de", "del", "desde", "donde", "dos", "durante", "e", "el", "él", "ella", "ellas", "ellos", "en", "entre",
    "era", "eran", "eres", "es", "esa", "esas", "ese", "eso", "esos", "esta", "está", "estaba", "estaban", "estado", "estamos", "estan", "están",
    "estar", "este", "esto", "estos", "estoy", "fue", "fueron", "fui", "ha", "haber", "habia", "había", "hace", "hacen", "hacer", "hacia",
    "han", "hasta", "hay", "la", "las", "le", "les", "lo", "los", "mas", "más", "me", "mi", "mis", "mismo", "mucha", "muchas", "mucho", "muchos",
    "muy", "nada", "ni", "no", "nos", "nosotras", "nosotros", "nuestra", "nuestras", "nuestro", "nuestros", "o", "otra", "otras", "otro", "otros",
    "para", "pero", "poco", "por", "porque", "que", "qué", "quien", "quién", "se", "sea", "segun", "según", "ser", "si", "sí", "sin",
    "sobre", "son", "su", "sus", "tambien", "también", "tan", "tanto", "te", "tenia", "tenía", "tener", "tiene", "tienen", "todo", "todos",
    "tu", "tus", "un", "una", "unas", "uno", "unos", "usted", "ustedes", "va", "van", "vez", "y", "ya", "yo",
}

DOMAIN_STOPWORDS = {
    "anfibia", "revista", "cronica", "crónica", "texto", "nota", "parte", "dice", "dijo", "hacer", "hace", "anos", "años",
    "argentina", "argentino", "argentinas", "argentinos", "buenos", "aires", "ano", "año", "poder",
    "decir", "ir", "ver", "solo", "llegar", "saber", "dar", "querer", "pasar", "volver", "día", "nuevo", "parecer", "quedar", 
    "creer", "dejar", "seguir", "encontrar", "llamar", "venir", "empezar", "persona", "historia", "vida", "caso", "momento", 
    "tiempo", "lugar", "forma", "primero", "despues", "después", "hoy", "ahora", "siempre", "nunca", "mientras", "cosa", 
    "alguno", "algun", "medio", "propio", "cierto", "grande"
}

STOPWORDS = sorted(SPANISH_STOPWORDS | DOMAIN_STOPWORDS)

# Preparamos las dos versiones
docs_naturales = df["document_text_original"].tolist()
docs_lematizados = df["document_text_final"].tolist()

# Fechas
timestamps = pd.to_datetime(df["post_date"].str.split(' ').str[0], format="%Y-%m-%d", errors="coerce")

## 4. Entrenamiento del Modelo Híbrido

Acá ocurre la separación de la arquitectura:
1. `SentenceTransformer` calcula los embeddings usando el **texto natural**.
2. `CountVectorizer` construye el vocabulario usando los **lemas**.
3. `BERTopic.fit_transform()` recibe los textos lematizados para el c-TF-IDF, pero se le inyectan los `embeddings_naturales` precalculados para que el clustering (UMAP + HDBSCAN) se haga sobre el espacio semántico original.

In [9]:
# 1. Embeddings sobre texto crudo
embedding_model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
embeddings_naturales = embedding_model.encode(docs_naturales, show_progress_bar=True)

# 2. Vectorizador sobre texto lematizado
vectorizer_model = CountVectorizer(stop_words=STOPWORDS, min_df=3)
representation_model = KeyBERTInspired()

topic_model = BERTopic(
    embedding_model=embedding_model,
    language="spanish",
    calculate_probabilities=True,
    min_topic_size=10,
    nr_topics="auto",
    representation_model=representation_model,
    vectorizer_model=vectorizer_model,
)

# 3. Entrenamiento: Clusteriza con embeddings naturales, extrae palabras con lemas
try:
    topic_model.load(OUTPUT_DIR / "bertopic_hibrido_model")
    topics, probs = topic_model.transform(docs_lematizados, embeddings=embeddings_naturales)
    print("Modelo cargado desde disco.")
except:
    topics, probs = topic_model.fit_transform(docs_lematizados, embeddings=embeddings_naturales)
    topic_model.save(OUTPUT_DIR / "bertopic_hibrido_model")
    print("Modelo híbrido entrenado y guardado.")

topic_info = topic_model.get_topic_info()
display(topic_info.head(10))

Batches: 100%|██████████| 112/112 [01:32<00:00,  1.21it/s]
2026-08-26 20:37:02,617 - BERTopic - WARNING: When you use `pickle` to save/load a BERTopic model,please make sure that the environments in which you saveand load the model are **exactly** the same. The version of BERTopic,its dependencies, and python need to remain the same.


Modelo híbrido entrenado y guardado.


,Topic,Count,Name,Representation,Representative_Docs
0,-1,1405,-1_político_política_junto_contar,"[político, política, junto, contar, además, so...",[el lucha por el legitimidad de el palabra en ...
1,0,1686,0_político_política_discurso_junto,"[político, política, discurso, junto, sociedad...",[del pánico a el revancha desde su nacimiento ...
2,1,113,1_noticia_sociedad_tecnológico_periodismo,"[noticia, sociedad, tecnológico, periodismo, t...",[el mapa de medio en el era milei a diferencia...
3,2,86,2_aborto_abortar_anticonceptivo_embarazo,"[aborto, abortar, anticonceptivo, embarazo, em...",[evita no usar pañuelo en marzo el campaña por...
4,3,75,3_pandemia_epidemia_vacunar_vacuna,"[pandemia, epidemia, vacunar, vacuna, vacunaci...",[ser más frágil de él que pensár el covid-19 p...
5,4,46,4_papal_catolicismo_arzobispo_católica,"[papal, catolicismo, arzobispo, católica, cató...",[el último hombre bueno hacer 12 año cuando jo...
6,5,30,5_arte_art_estética_artístico,"[arte, art, estética, artístico, artista, esté...",[bajar el cuadro subir el plasma entre 80 y 85...
7,6,27,6_soviética_soviético_vladimir_rusia,"[soviética, soviético, vladimir, rusia, ucrani...",[el paz ser uno sueño eterno a uno año del con...
8,7,19,7_israel_netanyahu_israelí_palestina,"[israel, netanyahu, israelí, palestina, palest...",[herida de uno cuento inventado cada vez más v...
9,8,16,8_cientificismo_astrología_universo_ciencia,"[cientificismo, astrología, universo, ciencia,...",[científicas y mágico en todo el ámbito de el ...


In [10]:
topics_over_time = topic_model.topics_over_time(
    docs_lematizados,
    timestamps.tolist(),
    datetime_format="%Y-%m-%d",
    nr_bins=20,
)
topics_over_time.to_csv(OUTPUT_DIR / "topics_over_time.csv", index=False)

analysis_df = df[['post_date', 'category']].copy()
analysis_df['category'] = categories_df['category_normalized']
analysis_df['category'] = analysis_df['category'].fillna('sin_categoria').astype(str).str.strip().replace('', 'sin_categoria')
analysis_df['post_date'] = pd.to_datetime(analysis_df['post_date'], errors='coerce')
analysis_df['topic'] = topics

topics_by_category = (
    analysis_df
    .assign(date=lambda x: x['post_date'].dt.to_period('M').dt.to_timestamp())
    .groupby(['date', 'category', 'topic'])
    .size()
    .reset_index(name='count')
)
topics_by_category = topics_by_category[topics_by_category['topic'] != -1].copy()
topics_by_category.to_csv(OUTPUT_DIR / 'topics_over_time_by_category.csv', index=False)

## 5. Extracción de la evolución temporal

In [11]:
fig = topic_model.visualize_barchart(top_n_topics=10, n_words=8, title="<b>Palabras representativas (Híbrido)</b>")
colors = px.colors.qualitative.G10
for i, trace in enumerate(fig.data):
    trace.marker.color = colors[i % len(colors)]

fig.update_traces(hovertemplate="<b>Término:</b> %{y}<br><b>Relevancia:</b> %{x:.3f}<extra></extra>")
fig.for_each_xaxis(lambda ax: ax.update(title_text="", showticklabels=False))
fig.for_each_annotation(lambda a: a.update(text=f"<b>{a.text.split('_')[0]}</b>"))
fig.update_layout(height=750, width=1300, plot_bgcolor='rgba(0,0,0,0)', paper_bgcolor='#ffffff')

fig.write_html(GRAFICOS_DIR / "01_barchart_topicos.html")
fig.show()

## 6. Visualizaciones y Diagnóstico

### Palabras más representativas (Barchart)

In [12]:
fig = topic_model.visualize_topics_over_time(topics_over_time, top_n_topics=10)
colors = px.colors.qualitative.G10

for i, t in enumerate(list(fig.data)):
    c = colors[i % len(colors)]
    vis = True if i < 3 else 'legendonly'
    name = t.name.replace("_", " ")

    t.update(name=name, visible=vis, legendgroup=f"g{i}", line=dict(color=c, width=3),
             hovertemplate=f"<b>{name}</b><br>Fecha: %{{x}}<br>Frecuencia: %{{y}}<extra></extra>")
    
    idx = list(t.y).index(max(t.y))
    fig.add_trace(go.Scatter(
        x=[t.x[idx]], y=[t.y[idx]], text=[t.y[idx]], mode="markers+text", textposition="top center",
        showlegend=False, visible=vis, legendgroup=f"g{i}",
        marker=dict(size=10, color=c, symbol="diamond", line=dict(width=1.5, color="#111")),
        hovertemplate=f"<b>Pico ({name})</b><br>Fecha: %{{x}}<br>Máximo: %{{y}}<extra></extra>"
    ))

fig.update_layout(title="<b>Evolución temporal de Tópicos</b>", plot_bgcolor='rgba(0,0,0,0)', paper_bgcolor='#fff', height=600, hovermode="closest")
fig.update_yaxes(showgrid=True, gridcolor="#ebebeb")

fig.write_html(GRAFICOS_DIR / "02_evolucion_topicos_picos.html")
fig.show()

### Evolución temporal de los tópicos principales

In [13]:
# Grafica proyectando los embeddings originales pero etiquetando con las palabras clave lematizadas
fig = topic_model.visualize_documents(docs_lematizados, embeddings=embeddings_naturales)

fig.update_layout(title="<b>Espacio de documentos (Embeddings Híbridos)</b>", plot_bgcolor='rgba(0,0,0,0)', paper_bgcolor='#fff', height=600, hovermode="closest")
fig.update_yaxes(showgrid=True, gridcolor="#ebebeb")

fig.write_html(GRAFICOS_DIR / "03_espacio_documentos.html")
fig.show()

## Evaluo como estan distribuidos los topicos ahora


In [15]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# 1. Matriz de coocurrencia de términos en el corpus
vectorizer = CountVectorizer(vocabulary=topic_model.vectorizer_model.vocabulary_)
X = vectorizer.transform(docs_lematizados)
binary_matrix = (X > 0).astype(int)

doc_counts = np.array(binary_matrix.sum(axis=0)).flatten()
n_docs = binary_matrix.shape[0]

# 2. Función para calcular coherencia NPMI por tópico
def calculate_topic_npmi(words, top_n=10):
    words = words[:top_n]
    vocab = vectorizer.vocabulary_
    indices = [vocab[w] for w in words if w in vocab]
    
    if len(indices) < 2:
        return 0.0
    
    npmi_scores = []
    for i in range(len(indices)):
        for j in range(i + 1, len(indices)):
            idx_i, idx_j = indices[i], indices[j]
            p_i = doc_counts[idx_i] / n_docs
            p_j = doc_counts[idx_j] / n_docs
            
            # Documentos donde coocurren ambas palabras
            col_i = binary_matrix[:, idx_i].toarray().flatten()
            col_j = binary_matrix[:, idx_j].toarray().flatten()
            cooc = np.sum(col_i & col_j)
            p_ij = cooc / n_docs
            
            if p_ij > 0:
                pmi = np.log2(p_ij / (p_i * p_j))
                npmi = pmi / (-np.log2(p_ij))
                npmi_scores.append(npmi)
            else:
                npmi_scores.append(-1.0)
                
    return np.mean(npmi_scores) if npmi_scores else 0.0

# 3. Métricas consolidadas por tópico
topic_metrics = []
topics_dict = topic_model.get_topics()

for t_id, words_weights in topics_dict.items():
    if t_id == -1:
        continue
    words = [w for w, _ in words_weights]
    count = topic_model.get_topic_info(t_id)["Count"].values[0]
    pct = (count / n_docs) * 100
    npmi = calculate_topic_npmi(words)
    
    topic_metrics.append({
        "Topic": t_id,
        "Name": ", ".join(words[:4]),
        "Docs": count,
        "% Corpus": round(pct, 2),
        "Coherencia_NPMI": round(npmi, 4)
    })

metrics_df = pd.DataFrame(topic_metrics).sort_values(by="Docs", ascending=False)
display(metrics_df.head(15))

,Topic,Name,Docs,% Corpus,Coherencia_NPMI
0,0,"político, política, discurso, junto",1686,47.41,0.0895
1,1,"noticia, sociedad, tecnológico, periodismo",113,3.18,0.1024
2,2,"aborto, abortar, anticonceptivo, embarazo",86,2.42,0.2436
3,3,"pandemia, epidemia, vacunar, vacuna",75,2.11,0.3503
4,4,"papal, catolicismo, arzobispo, católica",46,1.29,0.4778
5,5,"arte, art, estética, artístico",30,0.84,0.2240
6,6,"soviética, soviético, vladimir, rusia",27,0.76,0.4248
7,7,"israel, netanyahu, israelí, palestina",19,0.53,0.4322
8,8,"cientificismo, astrología, universo, ciencia",16,0.45,0.0838
9,9,"submarino, marino, océano, naval",15,0.42,0.2331


* **% Corpus (% del total de notas):**
  * **$< 0.5\% - 0.8\%$:** Tópicos marginales o notas aisladas.
  * **$\ge 1.0\%$:** Tópicos con volumen representativo dentro del corpus.

* **Coherencia NPMI (rango de -1 a +1):**
Mide qué tan fuertemente asociadas están dos palabras en un corpus.
  * **$> 0.10$:** Excelente coherencia semántica. Las palabras clave aparecen frecuentemente juntas en los mismos contextos.
  * **$0.00$ a $0.10$:** Aceptable. Tópico amplio o general, pero interpretable.
  * **$< -0.05$:** Tópico ruidoso o inconexo. Candidato directo a descartar o fusionar.